# 面试问题：线上模型的数据漂移、预测漂移和性能退化应该怎样监控？

**一句话回答**：分别监控输入、模型输出和延迟标签性能；baseline 必须绑定模型/特征版本与时间窗口。数值特征用固定 baseline bins 的 PSI/JS/KS，类别特征监控新类别与频率，流式均值用 Page-Hinkley；所有告警按业务切片、样本量和连续窗口做门禁，漂移只触发调查，不等于模型一定坏了。

下面只用 NumPy 从零实现这些统计量、延迟标签 join、分组切片、迟滞告警和监控 manifest。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
import hashlib, json, math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED87=8701; rng87=np.random.default_rng(SEED87)  # 计算并保存当前步骤的中间状态。
baseline87=rng87.normal(0,1,5000); stable87=rng87.normal(.05,1.02,3000); shifted87=rng87.normal(.8,1.3,3000)  # 计算并保存当前步骤的中间状态。
assert len(baseline87)==5000 and np.isfinite(baseline87).all()  # 用受控断言验证关键不变量。
assert abs(baseline87.mean())<.1  # 用受控断言验证关键不变量。
assert shifted87.mean()>stable87.mean()+.5  # 用受控断言验证关键不变量。

## 1. Baseline 和窗口合同

baseline 不是“历史全部数据”，而是已验证模型在某版本、地区、业务季节下的参考分布；live window 保存 event-time 范围、样本量、缺失率和数据质量结果。训练分布、上周分布和同星期分布回答不同问题。

bins 只能由 baseline 拟合，再固定应用于 live；若分别按两边 quantile 分桶，两个分布都会近似均匀，漂移被抹掉。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Window87:  # 定义承载本节状态与行为的数据结构。
    feature_version:str; start:int; end:int; values:np.ndarray  # 执行当前语句以推进本节示例。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        v=np.asarray(self.values)  # 计算并保存当前步骤的中间状态。
        if not self.feature_version or self.end<=self.start or v.ndim!=1 or len(v)<20: raise ValueError("window_contract")  # 按当前条件选择后续控制路径。
w87=Window87("feature-v3",0,60,stable87)  # 计算并保存当前步骤的中间状态。
assert w87.end-w87.start==60 and len(w87.values)==3000  # 用受控断言验证关键不变量。
try: Window87("",1,1,np.ones(30)); raise AssertionError("invalid window accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="window_contract"  # 捕获预期异常并验证失败分支。
assert np.isnan([1.,np.nan]).sum()==1  # 用受控断言验证关键不变量。

## 2. 从零实现 PSI

用 baseline quantile 生成内部边界，并扩到 `[-inf,+inf]`。每 bin 比例加 epsilon 后归一化，`PSI=Σ(q-p)log(q/p)`。PSI 对分桶和样本量敏感，没有跨业务通用的 0.1/0.2 魔法阈值；阈值应通过历史稳定窗口和已知故障校准。

相同分布 PSI 接近 0，位置/尺度变化会升高。

In [ ]:
def baseline_bins87(x,bins=10):  # 定义本节可复用的核心函数。
    x=np.asarray(x,float); x=x[np.isfinite(x)]  # 计算并保存当前步骤的中间状态。
    if len(x)<bins or bins<2: raise ValueError("bin_contract")  # 按当前条件选择后续控制路径。
    inner=np.unique(np.quantile(x,np.linspace(0,1,bins+1)[1:-1])); return np.r_[-np.inf,inner,np.inf]  # 计算并保存当前步骤的中间状态。
def proportions87(x,edges,eps=1e-6):  # 定义本节可复用的核心函数。
    x=np.asarray(x,float); finite=x[np.isfinite(x)]; counts=np.histogram(finite,bins=edges)[0].astype(float)+eps; return counts/counts.sum()  # 计算并保存当前步骤的中间状态。
def psi87(reference,current,edges):  # 定义本节可复用的核心函数。
    p=proportions87(reference,edges); q=proportions87(current,edges); return float(np.sum((q-p)*np.log(q/p)))  # 计算并保存当前步骤的中间状态。
edges87=baseline_bins87(baseline87,10); psi_stable87=psi87(baseline87,stable87,edges87); psi_shift87=psi87(baseline87,shifted87,edges87)  # 计算并保存当前步骤的中间状态。
assert len(edges87)==11 and np.all(np.diff(edges87)>0)  # 用受控断言验证关键不变量。
assert 0<=psi_stable87<.03 and psi_shift87>psi_stable87+.2  # 用受控断言验证关键不变量。
assert math.isclose(psi87(baseline87,baseline87,edges87),0.,abs_tol=1e-12)  # 用受控断言验证关键不变量。

## 3. JS divergence 与类别新值

JS 是对称、有界的分布差异：`0.5 KL(p||m)+0.5 KL(q||m)`。对类别特征建立 baseline vocabulary，live 中未知类别合并为 `__OTHER__`，同时单独报告 unknown rate；否则新类别可能被静默丢弃。

数值直方图也能计算 JS，但仍依赖固定 bins。

In [ ]:
def js87(p,q):  # 定义本节可复用的核心函数。
    p=np.asarray(p,float); q=np.asarray(q,float)  # 计算并保存当前步骤的中间状态。
    if p.shape!=q.shape or np.any(p<0) or np.any(q<0) or p.sum()<=0 or q.sum()<=0: raise ValueError("distribution_contract")  # 按当前条件选择后续控制路径。
    p=p/p.sum(); q=q/q.sum(); m=(p+q)/2  # 计算并保存当前步骤的中间状态。
    kl=lambda a,b: float(np.sum(np.where(a>0,a*np.log(a/b),0.)))  # 计算并保存当前步骤的中间状态。
    return .5*kl(p,m)+.5*kl(q,m)  # 返回当前分支计算出的结果。
p87=proportions87(baseline87,edges87); qs87=proportions87(stable87,edges87); qd87=proportions87(shifted87,edges87)  # 计算并保存当前步骤的中间状态。
assert math.isclose(js87(p87,p87),0.,abs_tol=1e-12)  # 用受控断言验证关键不变量。
assert 0<=js87(p87,qs87)<js87(p87,qd87)<=math.log(2)  # 用受控断言验证关键不变量。
vocab87={"mobile","web"}; live_cat87=["mobile"]*70+["web"]*20+["tv"]*10; unknown_rate87=np.mean([x not in vocab87 for x in live_cat87])  # 计算并保存当前步骤的中间状态。
assert math.isclose(unknown_rate87,.1)  # 用受控断言验证关键不变量。

## 4. 手写 two-sample KS statistic

KS 是两条经验 CDF 的最大垂直距离，对一维连续特征敏感，不需分桶。这里合并排序后的唯一取值，用 `searchsorted(..., side='right')/n` 计算两边 CDF。它给出 effect size；显著性还取决于样本量，大流量下微小无害差异也会“显著”。

监控应同时报 KS 与业务阈值，不只报 p-value。

In [ ]:
def ks_stat87(a,b):  # 定义本节可复用的核心函数。
    a=np.sort(np.asarray(a,float)); b=np.sort(np.asarray(b,float)); a=a[np.isfinite(a)]; b=b[np.isfinite(b)]  # 计算并保存当前步骤的中间状态。
    if not len(a) or not len(b): raise ValueError("ks_contract")  # 按当前条件选择后续控制路径。
    grid=np.unique(np.r_[a,b]); fa=np.searchsorted(a,grid,side="right")/len(a); fb=np.searchsorted(b,grid,side="right")/len(b)  # 计算并保存当前步骤的中间状态。
    return float(np.max(np.abs(fa-fb)))  # 返回当前分支计算出的结果。
ks_stable87=ks_stat87(baseline87,stable87); ks_shift87=ks_stat87(baseline87,shifted87)  # 计算并保存当前步骤的中间状态。
assert 0<=ks_stable87<ks_shift87<=1  # 用受控断言验证关键不变量。
assert math.isclose(ks_stat87(baseline87,baseline87),0.)  # 用受控断言验证关键不变量。
try: ks_stat87([],stable87); raise AssertionError("empty accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="ks_contract"  # 捕获预期异常并验证失败分支。

## 5. Page-Hinkley 监控流式均值变化

对每个样本更新运行均值，累计 `x-mean-delta`，当累计量与历史最小值差超过 lambda 报警。delta 忽略微小变化，lambda 控制灵敏度。检测后可 reset；若不 reset，会持续重复告警。

它假设观测顺序可靠；乱序事件应先按 event time/watermark 聚合。

In [ ]:
class PageHinkley87:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,delta=.02,threshold=15): self.delta=delta; self.threshold=threshold; self.n=0; self.mean=0.; self.cum=0.; self.min_cum=0.; self.detected_at=None  # 定义本节可复用的核心函数。
    def update(self,x):  # 定义本节可复用的核心函数。
        self.n+=1; self.mean+=(x-self.mean)/self.n; self.cum+=x-self.mean-self.delta; self.min_cum=min(self.min_cum,self.cum)  # 计算并保存当前步骤的中间状态。
        alarm=self.cum-self.min_cum>self.threshold  # 计算并保存当前步骤的中间状态。
        if alarm and self.detected_at is None: self.detected_at=self.n  # 按当前条件选择后续控制路径。
        return alarm  # 返回当前分支计算出的结果。
stream87=np.r_[rng87.normal(0,1,400),rng87.normal(1,1,400)]; ph87=PageHinkley87(.02,25); alarms87=[ph87.update(float(x)) for x in stream87]  # 计算并保存当前步骤的中间状态。
assert ph87.detected_at is not None and 350<ph87.detected_at<650  # 用受控断言验证关键不变量。
assert any(alarms87) and ph87.n==800  # 用受控断言验证关键不变量。
stable_ph87=PageHinkley87(.02,30); stable_alarms87=[stable_ph87.update(float(x)) for x in rng87.normal(0,1,500)]  # 计算并保存当前步骤的中间状态。
assert sum(stable_alarms87)<20  # 用受控断言验证关键不变量。

## 6. 延迟标签：prediction drift 不等于 performance drift

特征/预测可实时监控，accuracy/AUC 需标签到达后按 request ID join。标签可能只覆盖部分样本且延迟与结果相关，直接在“已回流标签”上算指标会选择偏差。必须报告 label coverage、delay distribution 和 join failure。

这里用 Brier score 演示版本化 join，并拒绝重复/未知 label。

In [ ]:
predictions87={f"r{i}":{"p":float(p),"model":"m2","group":"a" if i%3 else "b"} for i,p in enumerate(rng87.uniform(.05,.95,200))}  # 计算并保存当前步骤的中间状态。
labels87={f"r{i}":int(rng87.random()<predictions87[f"r{i}"]["p"]) for i in range(150)}  # 计算并保存当前步骤的中间状态。
joined87=[(predictions87[r]["p"],y,predictions87[r]["group"]) for r,y in labels87.items() if r in predictions87]  # 计算并保存当前步骤的中间状态。
brier87=float(np.mean([(p-y)**2 for p,y,_ in joined87])); coverage87=len(joined87)/len(predictions87)  # 计算并保存当前步骤的中间状态。
assert 0<=brier87<=1 and math.isclose(coverage87,.75)  # 用受控断言验证关键不变量。
assert len(joined87)==len(labels87)==150  # 用受控断言验证关键不变量。
group_brier87={g:float(np.mean([(p-y)**2 for p,y,gg in joined87 if gg==g])) for g in {x[2] for x in joined87}}  # 计算并保存当前步骤的中间状态。
assert set(group_brier87)=={"a","b"} and all(0<=v<=1 for v in group_brier87.values())  # 用受控断言验证关键不变量。

## 7. 样本门禁、连续窗口和迟滞

单个窗口越线不应立刻重训。状态机要求样本量达标且连续 `N` 个窗口超阈值才 OPEN；恢复则要求更低的 clear threshold 连续若干窗口，形成 hysteresis，避免告警抖动。数据质量故障与真实漂移用不同 reason code。

自动动作优先降级/回滚或启动调查；重训还需标签、评估和审批。

In [ ]:
class Alert87:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,open_threshold=.2,clear_threshold=.08,consecutive=2,min_n=500): self.open_t=open_threshold; self.clear_t=clear_threshold; self.consecutive=consecutive; self.min_n=min_n; self.state="CLOSED"; self.bad=0; self.good=0  # 定义本节可复用的核心函数。
    def update(self,value,n):  # 定义本节可复用的核心函数。
        if n<self.min_n: return self.state,"insufficient_sample"  # 按当前条件选择后续控制路径。
        if self.state=="CLOSED":  # 按当前条件选择后续控制路径。
            self.bad=self.bad+1 if value>=self.open_t else 0  # 计算并保存当前步骤的中间状态。
            if self.bad>=self.consecutive: self.state="OPEN"; self.good=0  # 按当前条件选择后续控制路径。
        else:  # 处理前置条件不成立的分支。
            self.good=self.good+1 if value<=self.clear_t else 0  # 计算并保存当前步骤的中间状态。
            if self.good>=self.consecutive: self.state="CLOSED"; self.bad=0  # 按当前条件选择后续控制路径。
        return self.state,"evaluated"  # 返回当前分支计算出的结果。
alert87=Alert87(); assert alert87.update(.5,100)[1]=="insufficient_sample"  # 计算并保存当前步骤的中间状态。
assert alert87.update(.25,1000)[0]=="CLOSED" and alert87.update(.3,1000)[0]=="OPEN"  # 用受控断言验证关键不变量。
assert alert87.update(.05,1000)[0]=="OPEN" and alert87.update(.04,1000)[0]=="CLOSED"  # 用受控断言验证关键不变量。

## 8. Manifest、切片与面试收束

monitor manifest 绑定模型、特征 schema、baseline 时间、bins/category vocabulary、窗口/watermark、指标实现和 alert thresholds。模型升级后不能继续沿用旧预测 baseline。仪表盘至少按 tenant/region/device/model version 切片，并设最小样本门禁。

完整回答：三类漂移 → baseline/window 合同 → PSI/JS/KS/Page-Hinkley → 延迟标签 → 切片/样本量 → 迟滞告警 → 调查、回滚与重训闭环。

In [ ]:
manifest87={"schema":1,"model":"m2","feature_version":"feature-v3","baseline":"2026-w20","edges":[float(x) for x in edges87],"metrics":["psi","js","ks","page_hinkley","brier"],"alert":{"open":.2,"clear":.08,"consecutive":2}}  # 计算并保存当前步骤的中间状态。
raw87=json.dumps(manifest87,sort_keys=True,separators=(",",":")); digest87=hashlib.sha256(raw87.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert len(digest87)==64 and len(manifest87["edges"])==11  # 用受控断言验证关键不变量。
assert psi_shift87>manifest87["alert"]["open"] and psi_stable87<manifest87["alert"]["clear"]  # 用受控断言验证关键不变量。
assert manifest87["model"]==next(iter(predictions87.values()))["model"]  # 用受控断言验证关键不变量。
print({"psi_stable":round(psi_stable87,3),"psi_shift":round(psi_shift87,3),"ks_shift":round(ks_shift87,3),"coverage":coverage87})  # 执行当前语句以推进本节示例。

## 9. 参考与练习

练习：实现类别 PSI；用历史稳定日 bootstrap alert threshold；加入 event-time watermark；模拟标签只对高分样本回流造成的偏差；设计 drift alert 到 shadow retrain 的审批状态机。

参考：[Page-Hinkley 原始方法综述入口](https://link.springer.com/book/10.1007/978-1-4757-3261-5)、[NIST KS 检验说明](https://www.itl.nist.gov/div898/handbook/eda/section3/eda35g.htm)、[Google Rules of ML：监控训练与服务偏差](https://developers.google.com/machine-learning/guides/rules-of-ml)。